# 🎯 Yuzu Memory Rebuild - GPU Edition

**Runtime:** Google Colab T4 GPU (FREE)
**Speed:** ~5-10 menit untuk 7000+ messages

**Workflow:**
1. 🔐 Setup secrets
2. 📥 Pull dari Supabase
3. 🧠 Embed dengan GPU (e5-base)
4. 📤 Push balik ke Supabase

## 🔐 Step 1: Secrets Setup

Klik 🔑 (Secrets) di sidebar kiri, tambahkan:
- `SUPABASE_URL` = postgresql://...
- `SUPABASE_KEY` = eyJhbG...

In [ ]:
# Load secrets
from google.colab import userdata
import os

SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_KEY')

print(f"✅ URL: {SUPABASE_URL[:40]}..." if SUPABASE_URL else "❌ SUPABASE_URL not set")

## 📦 Step 2: Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q sentence-transformers psycopg2-binary pandas numpy
!pip install -q duckdb

import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 🧠 Step 3: Load Model (GPU)

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'intfloat/multilingual-e5-base'
EMBEDDING_DIM = 768

print(f"📥 Downloading {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME, device='cuda')
print(f"✅ Model loaded: {MODEL_NAME}")
print(f"📊 Embedding dimension: {EMBEDDING_DIM}")

## 📥 Step 4: Pull from Supabase

In [ ]:
import psycopg2
import pandas as pd
from tqdm import tqdm

# Connect
conn = psycopg2.connect(SUPABASE_URL, sslmode='require')
print("✅ Connected to Supabase")

# Pull messages
query = '''
SELECT m.id, m.session_id, m.role, m.content, m.created_at
FROM messages m
JOIN chat_sessions cs ON m.session_id = cs.id
WHERE cs.user_id = 1
ORDER BY m.created_at
'''

df = pd.read_sql(query, conn)
print(f"📥 Pulled {len(df)} messages")
conn.close()

# Preview
df.head()

## 🧠 Step 5: Generate Embeddings (GPU)

In [ ]:
# Filter valid messages
valid_df = df[df['content'].str.len() > 10].copy()
print(f"📝 Processing {len(valid_df)} valid messages...")

# Batch embed with GPU
BATCH_SIZE = 128
embeddings = []

for i in tqdm(range(0, len(valid_df), BATCH_SIZE)):
    batch = valid_df.iloc[i:i+BATCH_SIZE]['content'].tolist()
    emb = model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
    embeddings.extend(emb)

valid_df['embedding'] = embeddings
valid_df['memory_type'] = 'episodic'
valid_df['importance'] = 50
valid_df['user_id'] = 1

print(f"✅ Embedded {len(valid_df)} memories ({EMBEDDING_DIM} dims)")

## 📤 Step 6: Push to Supabase

In [ ]:
conn = psycopg2.connect(SUPABASE_URL, sslmode='require')
cur = conn.cursor()

BATCH_SIZE = 100
memories = valid_df.to_dict('records')

print(f"📤 Pushing {len(memories)} memories to Supabase...")

for i in tqdm(range(0, len(memories), BATCH_SIZE)):
    batch = memories[i:i+BATCH_SIZE]
    args = []
    for m in batch:
        emb_list = m['embedding'].tolist()
        args.extend([m['id'], m['user_id'], m['session_id'], m['content'], emb_list, m['memory_type'], m['importance']])
    
    values = ','.join(['(%s,%s,%s,%s,%s::vector(768),%s,%s)'] * len(batch))
    cur.execute(f'''
        INSERT INTO memories (id, user_id, session_id, content, embedding, memory_type, importance)
        VALUES {values}
        ON CONFLICT DO NOTHING
    ''', args)
    conn.commit()

cur.close()
conn.close()

print(f"🎉 Done! Pushed {len(memories)} memories to Supabase")

## ✅ Complete!

Your ~7000 messages are now embedded with e5-base (768 dims) and stored in Supabase.